In [ ]:
### Inspect drug candidates

from common import *
from tqdm.auto import tqdm
import openbabel
import pickle

from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

from IPDiff.utils.visualize import visualize_protein_ligand
import IPDiff.utils.transforms as trans
from IPDiff.utils import reconstruct
from IPDiff.utils import misc

protein_path = root_dir + '/pockets/7upg_pocket1.pdb'

with open(os.path.join(root_dir, 'candidates_all.pkl'), 'rb') as f:
    candidates_all = pickle.load(f)

In [ ]:
for idx, (ligand_pred, similarity, mol) in enumerate(candidates_all):
    if mol is not None:
        with open(protein_path, 'r') as f:
            pdb_block = f.read()
        sdf_block = Chem.MolToMolBlock(ligand_pred)

        vis = visualize_protein_ligand(pdb_block, sdf_block, show_ligand=True, show_surface=True)
        vis.show()
        
        if idx > 10:
            break

In [ ]:
from IPDiff.utils.evaluation.docking_vina import VinaDockingTask

def convert_pdbqt_to_pdb(pdbqt_file, pdb_file):
    obConversion = openbabel.OBConversion()
    # Set input and output formats
    obConversion.SetInAndOutFormats("pdbqt", "pdb")
    
    mol = openbabel.OBMol()
    if obConversion.ReadFile(mol, pdbqt_file):
        obConversion.WriteFile(mol, pdb_file)
    else:
        print("Failed to read the PDBQT file.")

def visualize_similar_mols(mol1, mol2):
    # compute 3D coordinates
    pos1, pos2 = mol1.GetConformer().GetPositions(), mol2.GetConformer().GetPositions()
    
    # centralize both molecules
    pos1 -= pos1.mean(axis=0)
    pos2 -= pos2.mean(axis=0)
    
    # pca to rotate both molecules to the same direction
    from sklearn.decomposition import PCA
    pca = PCA(n_components=3)
    pca.fit(pos1)
    pos1 = pos1 @ pca.components_.T
    pca.fit(pos2)
    pos2 = pos2 @ pca.components_.T
    
    # create offset along the second principal component
    dist = np.max(pos1, axis=0) - np.min(pos2, axis=0)
    offset = np.array([0, dist[1] + 3, 0])
    pos2 += offset
    
    mol1.GetConformer().SetPositions(pos1)
    mol2.GetConformer().SetPositions(pos2)
    
    import py3Dmol
    view = py3Dmol.view()
    view.addModel(Chem.MolToMolBlock(mol1), 'sdf')
    view.setStyle({'model': -1}, {'stick': {}})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.8}, {'model': -1})
    view.addModel(Chem.MolToMolBlock(mol2), 'sdf')
    view.setStyle({'model': -1}, {'stick': {}})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.8}, {'model': -1})
    view.show()

for idx, (ligand_pred, similarity, mol) in enumerate(candidates_all):
    if mol is not None:
        with open(protein_path, 'r') as f:
            pdb_block = f.read()
        # sdf_block = Chem.MolToMolBlock(ligand_pred)

        # vis = visualize_protein_ligand(pdb_block, sdf_block, show_ligand=True, show_surface=True)
        # vis.show()

        vina_task = VinaDockingTask(protein_path, mol)
        docking_results = vina_task.run(exhaustiveness=32)
        affinity, pose = docking_results[0]['affinity'], docking_results[0]['pose']
        print(idx, similarity, affinity)
        
        if affinity > 0:
            # Convert PDBQT pose to PDB format
            os.makedirs('tmp', exist_ok=True)
            pdbqt_file = os.path.join('tmp', f'{idx}_{affinity}.pdbqt')
            pdb_file = os.path.join('tmp', f'{idx}_{affinity}.pdb')
            with open(pdbqt_file, 'w') as f:
                f.write(pose)
            convert_pdbqt_to_pdb(pdbqt_file, pdb_file)
            pdb_ligand = open(pdb_file, 'r').read()
            try:
                # Convert vina pose to sdf block
                mol = Chem.MolFromPDBBlock(pdb_ligand)
                sdf_block = Chem.MolToMolBlock(mol)
                visualize_protein_ligand(pdb_block, sdf_block, show_ligand=True, show_surface=True).show()
            except Exception as e:
                print(f"Error processing {idx}: {e}")